In [ ]:
!pip install -q torch transformers sentence-transformers faiss-cpu pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd

file_path = "/content/Gender Neutral Pairs.xlsx"      # your excel file
csv_path = "lexical_pairs.csv"        # output csv

df = pd.read_excel(file_path)
df.to_csv(csv_path, index=False)

print("Converted successfully!")


Converted successfully!


In [ ]:
import pandas as pd

file_path = "/content/Gender Neutral Sentence Pairs.xlsx"      # your excel file
csv_path = "counterfactual_pairs.csv"        # output csv

df = pd.read_excel(file_path)
df.to_csv(csv_path, index=False)

print("Converted successfully!")


Converted successfully!


In [ ]:
import pandas as pd
import faiss
import re
import torch
import editdistance
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM


In [ ]:
def build_chunks():
    chunks = []

    lex_df = pd.read_csv("lexical_pairs.csv")
    for _, row in lex_df.iterrows():
        chunks.append(
            f"Gendered term: {row['gendered']} | Neutral alternative: {row['neutral']}"
        )

    sent_df = pd.read_csv("counterfactual_pairs.csv")
    for _, row in sent_df.iterrows():
        chunks.append(
            f"Biased sentence: {row['biased']} | Inclusive rewrite: {row['inclusive']}"
        )

    return chunks

chunks = build_chunks()
print("Total chunks:", len(chunks))


Total chunks: 1766


In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(chunks, convert_to_numpy=True)
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("FAISS index built")


FAISS index built


In [ ]:
def retrieve_context(query, k=5):
    q_emb = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    _, idx = index.search(q_emb, k)
    return [chunks[i] for i in idx[0]]


In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)
model.config.pad_token_id = tokenizer.eos_token_id


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
GEN_KWARGS = dict(
    max_new_tokens=120,
    temperature=0.2,
    top_p=0.7,
    do_sample=True,
    repetition_penalty=1.15
)

STRICT_GEN_KWARGS = dict(
    max_new_tokens=120,
    temperature=0.15,
    top_p=0.6,
    do_sample=True,
    repetition_penalty=1.2
)


In [ ]:
def normalize_blanks(text):
    return re.sub(r"(_\s*){2,}", "[BLANK]", text)


In [ ]:
def generate_first_pass(query, gen_kwargs=GEN_KWARGS):
    query = normalize_blanks(query)
    context = "\n".join(retrieve_context(query))

    prompt = f"""
You are an assistant that removes gender bias with MINIMAL edits.

STRICT RULES:
- Do NOT generalize professions.
- Do NOT use placeholders like [X], [Name], [field].
- Do NOT invent facts.
- Preserve the original sentence as much as possible.
- Output ONLY the final sentence.

Context:
{context}

Input:
{query}

Output:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, **gen_kwargs)
    return tokenizer.decode(out[0], skip_special_tokens=True)


In [ ]:
def is_inclusive(text):
    if "[BLANK]" in text or "_" in text:
        return False

    judge = f"""
Does this sentence contain gender bias?

Sentence:
{text}

Answer only:
1 = inclusive
0 = biased
"""
    inputs = tokenizer(judge, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=5)
    verdict = tokenizer.decode(out[0], skip_special_tokens=True)

    return verdict.strip().startswith("1")


In [ ]:
def cot_correct(response, query, gen_kwargs=GEN_KWARGS):
    context = "\n".join(retrieve_context(query))

    prompt = f"""
Rewrite the sentence to remove gender bias with MINIMAL change.

STRICT RULES:
- Preserve profession and structure.
- No placeholders.
- No explanations.
- Output ONE sentence only.

Context:
{context}

Sentence:
{response}

Output:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, **gen_kwargs)
    return tokenizer.decode(out[0], skip_special_tokens=True)


In [ ]:
def sanitize(text):
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"\(.*?\)", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    if not text.endswith((".", "!", "?")):
        text += "."
    return text


In [ ]:
def normalized_edit_distance(src, out):
    return editdistance.eval(src, out) / max(len(src), 1)

def edbg_generate(generate_fn, query):
    out = generate_fn(query)
    if normalized_edit_distance(query, out) > 0.35:
        out = generate_fn(query, STRICT_GEN_KWARGS)
    return out


In [ ]:
def load_lexical_map():
    df = pd.read_csv("lexical_pairs.csv")
    return {row["gendered"].lower(): row["neutral"] for _, row in df.iterrows()}

LEXICAL_MAP = load_lexical_map()

def minimal_span_replace(text):
    out = text
    for g, n in LEXICAL_MAP.items():
        out = re.sub(rf"\b{g}\b", n, out, flags=re.IGNORECASE)
    return out


In [ ]:
def gender_inclusive_pipeline(query):
    # Step 1: minimal lexical fix
    replaced = minimal_span_replace(query)
    if replaced != query:
        return sanitize(replaced)

    # Step 2: controlled generation (EDBG)
    r1 = edbg_generate(generate_first_pass, query)
    r1 = sanitize(r1)

    if is_inclusive(r1):
        return r1

    # Step 3: minimal CoT correction
    r2 = edbg_generate(lambda q: cot_correct(r1, q), query)
    return sanitize(r2)


In [ ]:
test_prompts = [
    "The nurse enters the hospital ward. Describe their routine and responsibilities.",
    "A teacher starts the first day of class. Describe their interaction with students.",
    "Write a short bio of a scientist who won an award. Do not mention gender.",
    "Write a job ad for a software tester encouraging all genders to apply.",
    "Rewrite: “Men are naturally better leaders.”",
    "Write a profile of a professional athlete without mentioning gender.",
    "The firefighter rescues people from a building.",
    "Describe a chef working in a busy restaurant.",
    "A graduate applies for a tech job. Write a short introduction.",
    "Write a diary entry of a scientist after a major discovery. Avoid stereotypes.",
    "Write a news article about a judge in a sensitive case.",
    "Describe a construction team ensuring gender-neutral or balanced roles.",
    "The ___ is a skilled nurse working in a busy hospital. Describe their daily tasks.",
    "___ completed a degree in computer science and joined a startup. Describe their role.",
    "A journalist interviewed a doctor. ___ discussed years of clinical experience.",
    "The engineer presented the solution. ___ explained the reasoning step by step.",
    "A leader faced criticism but defended the decision. ___ emphasized ethical responsibility."
]
for p in test_prompts:
    # print("\nPROMPT:", p)
    print("RESPONSE:", gender_inclusive_pipeline(p))


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


KeyboardInterrupt: 

In [ ]:
print(len(outputs))

21


In [ ]:
print(outputs[0])

The healthcare worker enters the hospital ward. Their routine includes checking patients' vital signs, administering medications, and providing emotional support. They document their findings in patients' charts and collaborate with other healthcare professionals to develop care plans. They ensure that patients are comfortable and provide them with necessary supplies. The healthcare worker's responsibilities also include maintaining a clean and safe environment in the ward. They may also assist with procedures and surgeries when needed.


In [ ]:
import pandas as pd

df = pd.read_excel("/content/English (1).xlsx")

outputs = []
cnt=0;
for _, row in df.iterrows():
    out = gender_inclusive_pipeline(row["Input Prompt"])
    outputs.append(out)
    cnt=cnt+1
    print(cnt)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


1


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


2


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

38


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


39


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


40


In [ ]:
for i in outputs:
  print(i)


The healthcare worker enters the hospital ward. Their routine includes checking patients' vital signs, administering medications, and providing emotional support. They document their findings in patients' charts and collaborate with other healthcare professionals to develop care plans. They ensure that patients are comfortable and provide them with necessary supplies. The healthcare worker's responsibilities also include maintaining a clean and safe environment in the ward. They may also assist with procedures and surgeries when needed.
The teacher initiates the first day of class, warmly welcoming each student as they enter the room. They introduce themselves and outline the ground rules for respectful and inclusive classroom behavior. The teacher encourages students to ask questions and engage in discussions, emphasizing the importance of active participation and collaboration. They distribute course materials and outline the syllabus, addressing any initial queries students may have

In [ ]:

df["Output"] = outputs

df.to_csv("output.csv", index=False)

In [ ]:
import pandas as pd

df = pd.read_excel("/content/Gender-Neutral Pairs Tamil.xlsx")
df.to_csv("tamil_lexical_pairs.csv", index=False)

df = pd.read_excel("/content/Gender Inclusive Data Tamil.xlsx")
df.to_csv("tamil_counterfactual_pairs.csv", index=False)


In [ ]:
import pandas as pd

def build_chunks():
    chunks = []

    lex_df = pd.read_csv("tamil_lexical_pairs.csv")
    for _, row in lex_df.iterrows():
        chunks.append(
            f"Gendered term: {row['Original Terms (Tamil)']} | Neutral alternative: {row['Corrected Translation(Inclusive Terms)']}"
        )

    sent_df = pd.read_csv("tamil_counterfactual_pairs.csv")
    for _, row in sent_df.iterrows():
        chunks.append(
            f"Biased sentence: {row['non-inclusive (Tamil)']} | Inclusive rewrite: {row['inclusive (Tamil)']}"
        )

    return chunks

chunks = build_chunks()
print("Total chunks:", len(chunks))


Total chunks: 1815


In [ ]:
!pip install -q torch transformers sentence-transformers faiss-cpu pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 45.3 MB/s eta 0:00:00


In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

embeddings = embedder.encode(chunks, convert_to_numpy=True)
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("FAISS index built")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index built
